# 🏆 FIFA World Cup 2026 — Machine Learning Match Predictor
**Author:** Grigor Martirosyan

This notebook follows the Final Project guidelines:
1. Identify the problem type (Classification or Regression)
2. Explore and preprocess the data
3. Train ML models (Linear Regression, Logistic Regression, KNN)
4. Evaluate and visualize results
5. Simulate the FIFA World Cup 2026 tournament using our model

**Dataset:** Historical international football matches (1872–2026)

**Problem Type:** This is a **Classification** problem. We predict the outcome of a match: **Home Win (2)**, **Draw (1)**, or **Away Win (0)**.

In [ ]:
# Install required libraries
!pip install -q seaborn plotly

---
## 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ All libraries imported successfully')

---
## 📂 Step 1: Load the Dataset

Upload `all_matches.csv` and `countries_names.csv` from the sidebar (Files icon) or use the cell below.

In [ ]:
# If running in Colab, upload your files:
# from google.colab import files
# uploaded = files.upload()  # upload all_matches.csv and countries_names.csv

# Load datasets
matches_df = pd.read_csv('all_matches.csv')
countries_df = pd.read_csv('countries_names.csv')

print('✅ Datasets loaded!')
print(f'   Matches dataset: {matches_df.shape[0]:,} rows, {matches_df.shape[1]} columns')
print(f'   Countries dataset: {countries_df.shape[0]} rows, {countries_df.shape[1]} columns')

---
## 🔍 Step 2: Data Exploration
### 2.1 — Column Types

In [ ]:
print('=== COLUMN TYPES ===')
print(matches_df.dtypes)
print(f'\nTotal rows: {matches_df.shape[0]:,}')
print(f'Total columns: {matches_df.shape[1]}')
matches_df.head(10)

### 2.2 — Statistical Summary (mean, max, percentiles)

In [ ]:
print('=== STATISTICAL SUMMARY ===')
matches_df.describe()

### 2.3 — Missing Values Check

In [ ]:
print('=== MISSING VALUES PER COLUMN ===')
missing = matches_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')
print(f'\nTotal missing values: {matches_df.isnull().sum().sum()}')

### 2.4 — Correlation Between Columns

In [ ]:
print('=== CORRELATION MATRIX ===')
numeric_df = matches_df[['home_score', 'away_score']].copy()
numeric_df['neutral'] = matches_df['neutral'].astype(int)
corr = numeric_df.corr()
print(corr)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 — Visual Analysis: Goals Distribution & Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(matches_df['home_score'], bins=20, alpha=0.6, color='steelblue', label='Home Score')
axes[0].hist(matches_df['away_score'], bins=20, alpha=0.6, color='coral', label='Away Score')
axes[0].set_title('Goals Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Goals')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot (outlier detection)
axes[1].boxplot([matches_df['home_score'], matches_df['away_score']],
                labels=['Home Score', 'Away Score'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='navy'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Outlier Detection — Score Boxplot', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Goals')

plt.tight_layout()
plt.show()

print(f'Max home score: {matches_df["home_score"].max()} (outlier upper bound: {matches_df["home_score"].quantile(0.99):.0f})')
print(f'Max away score: {matches_df["away_score"].max()} (outlier upper bound: {matches_df["away_score"].quantile(0.99):.0f})')

---
## ⚙️ Step 3: Data Preprocessing & Feature Engineering
### 3.1 — Map Historical Country Names & Encode Categoricals

In [ ]:
# Fix countries_names.csv if it has duplicate comma on Vietnam row
# (already fixed in the CSV, but safe to re-parse with error_bad_lines=False)
countries_df = pd.read_csv('countries_names.csv', on_bad_lines='skip')

# Build country name mapping (old name -> current FIFA name)
country_map = dict(zip(countries_df['original_name'], countries_df['current_name']))

df = matches_df.copy()
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# Map names
df['home_team'] = df['home_team'].map(country_map).fillna(df['home_team'])
df['away_team'] = df['away_team'].map(country_map).fillna(df['away_team'])

# Drop rows with missing scores
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

# Encode neutral venue (bool -> int)
df['neutral'] = df['neutral'].astype(int)

# Target: score difference (regression)
df['score_diff'] = df['home_score'] - df['away_score']

# Target: outcome (classification): 2=Home Win, 1=Draw, 0=Away Win
df['outcome'] = np.where(df['score_diff'] > 0, 2, np.where(df['score_diff'] == 0, 1, 0))

print(f'✅ Dataset cleaned. Shape: {df.shape}')
print('\nOutcome distribution:')
print(df['outcome'].value_counts().rename({2:'Home Win', 1:'Draw', 0:'Away Win'}))

### 3.2 — Feature Engineering: Elo Rating System

In [ ]:
def get_k_factor(tournament):
    t = str(tournament).lower()
    if 'world cup' in t and 'qualifying' not in t: return 60
    elif any(x in t for x in ['copa', 'euro', 'african', 'asian cup']): return 50
    elif 'qualifying' in t or 'qualification' in t: return 40
    elif 'friendly' in t: return 20
    else: return 30

def calculate_elo_ratings(df, home_advantage=100):
    df = df.sort_values('date').reset_index(drop=True)
    elos = {}
    elo_home_before, elo_away_before = [], []

    for _, row in df.iterrows():
        home, away = row['home_team'], row['away_team']
        elos.setdefault(home, 1500.0)
        elos.setdefault(away, 1500.0)

        h_elo, a_elo = elos[home], elos[away]
        elo_home_before.append(h_elo)
        elo_away_before.append(a_elo)

        h_eff = h_elo + (0 if row['neutral'] else home_advantage)
        exp_h = 1.0 / (1.0 + 10 ** ((a_elo - h_eff) / 400.0))

        sd = row['home_score'] - row['away_score']
        actual_h = 1.0 if sd > 0 else (0.0 if sd < 0 else 0.5)

        k = get_k_factor(row['tournament'])
        elos[home] = h_elo + k * (actual_h - exp_h)
        elos[away] = a_elo + k * ((1 - actual_h) - (1 - exp_h))

    df['elo_home'] = elo_home_before
    df['elo_away'] = elo_away_before
    df['elo_diff'] = df['elo_home'] - df['elo_away']
    return df, elos

print('⏳ Calculating Elo ratings (this may take ~30 seconds)...')
df, latest_elos = calculate_elo_ratings(df)
print('✅ Elo ratings computed!')

### 3.3 — Feature Engineering: Recent Form

In [ ]:
def compute_form(df, span=5):
    df = df.sort_values('date').reset_index(drop=True)
    team_hist = {}
    form_home, form_away = [], []

    for _, row in df.iterrows():
        home, away = row['home_team'], row['away_team']
        for team in [home, away]:
            team_hist.setdefault(team, [])

        h_hist = team_hist[home][-span:]
        a_hist = team_hist[away][-span:]

        form_home.append(sum(m['s'] - m['c'] for m in h_hist) / len(h_hist) if h_hist else 0.0)
        form_away.append(sum(m['s'] - m['c'] for m in a_hist) / len(a_hist) if a_hist else 0.0)

        team_hist[home].append({'s': row['home_score'], 'c': row['away_score']})
        team_hist[away].append({'s': row['away_score'], 'c': row['home_score']})

    df['form_home'] = form_home
    df['form_away'] = form_away
    df['form_diff'] = df['form_home'] - df['form_away']
    return df

print('⏳ Computing form features...')
df = compute_form(df)
print('✅ Done!')

### 3.4 — Normalization & Final Feature Set

In [ ]:
# Filter to modern era (year >= 2000)
df_modern = df[df['year'] >= 2000].copy().reset_index(drop=True)

FEATURES = ['elo_home', 'elo_away', 'elo_diff', 'form_home', 'form_away', 'form_diff', 'neutral']

X = df_modern[FEATURES]
y_clf = df_modern['outcome']   # classification target
y_reg = df_modern['score_diff']  # regression target

# Train-test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# Normalize
scaler = StandardScaler()
X_train_c_s = scaler.fit_transform(X_train_c)
X_test_c_s  = scaler.transform(X_test_c)
X_train_r_s = scaler.transform(X_train_r)
X_test_r_s  = scaler.transform(X_test_r)

print(f'Training set size: {X_train_c.shape[0]:,} matches')
print(f'Test set size    : {X_test_c.shape[0]:,} matches')
print('\nFeatures after normalization (first 3 rows):')
pd.DataFrame(X_train_c_s[:3], columns=FEATURES)

---
## 🤖 Step 4: Model Training & Evaluation
### 4.1 — Linear Regression
> Predicts the **score difference** (home_score − away_score)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_r_s, y_train_r)
y_pred_lr = lr.predict(X_test_r_s)

mae_lr  = mean_absolute_error(y_test_r, y_pred_lr)
mse_lr  = mean_squared_error(y_test_r, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print('=== LINEAR REGRESSION ===')
print('Coefficients:')
for feat, coef in zip(FEATURES, lr.coef_):
    print(f'  {feat:20s}: {coef:+.4f}')
print(f'Intercept   : {lr.intercept_:.4f}')
print(f'\nMAE  = {mae_lr:.4f}')
print(f'MSE  = {mse_lr:.4f}')
print(f'RMSE = {rmse_lr:.4f}')

# Visualization
plt.figure(figsize=(10, 4))
sample = 200
plt.plot(y_test_r.values[:sample], label='Actual Score Diff', alpha=0.7)
plt.plot(y_pred_lr[:sample], label='Predicted Score Diff', alpha=0.7)
plt.title('Linear Regression — Actual vs Predicted Score Difference', fontweight='bold')
plt.xlabel('Match Index')
plt.ylabel('Score Difference')
plt.legend()
plt.tight_layout()
plt.show()

### 4.2 — Logistic Regression
> Predicts the **match outcome**: 0=Away Win, 1=Draw, 2=Home Win

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_c_s, y_train_c)
y_pred_log = log_reg.predict(X_test_c_s)

acc_log  = accuracy_score(y_test_c, y_pred_log)
prec_log = precision_score(y_test_c, y_pred_log, average='weighted', zero_division=0)
rec_log  = recall_score(y_test_c, y_pred_log, average='weighted', zero_division=0)

print('=== LOGISTIC REGRESSION ===')
print('Coefficients per class (Away Win | Draw | Home Win):')
coef_df = pd.DataFrame(log_reg.coef_.T, index=FEATURES,
                        columns=['Away Win', 'Draw', 'Home Win'])
print(coef_df.to_string())
print(f'\nIntercept: {log_reg.intercept_}')
print(f'\nAccuracy  = {acc_log:.4f}')
print(f'Precision = {prec_log:.4f}')
print(f'Recall    = {rec_log:.4f}')

# Confusion Matrix
cm_log = confusion_matrix(y_test_c, y_pred_log)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_log,
                               display_labels=['Away Win', 'Draw', 'Home Win'])
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Logistic Regression — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 — K-Nearest Neighbors (KNN)
> Both Classifier and Regressor variants

In [ ]:
# ── KNN Classifier
knn_clf = KNeighborsClassifier(n_neighbors=15)
knn_clf.fit(X_train_c_s, y_train_c)
y_pred_knn_c = knn_clf.predict(X_test_c_s)

acc_knn  = accuracy_score(y_test_c, y_pred_knn_c)
prec_knn = precision_score(y_test_c, y_pred_knn_c, average='weighted', zero_division=0)
rec_knn  = recall_score(y_test_c, y_pred_knn_c, average='weighted', zero_division=0)

# ── KNN Regressor
knn_reg = KNeighborsRegressor(n_neighbors=15)
knn_reg.fit(X_train_r_s, y_train_r)
y_pred_knn_r = knn_reg.predict(X_test_r_s)

mae_knn  = mean_absolute_error(y_test_r, y_pred_knn_r)
mse_knn  = mean_squared_error(y_test_r, y_pred_knn_r)
rmse_knn = np.sqrt(mse_knn)

print('=== KNN CLASSIFIER ===')
print(f'Accuracy  = {acc_knn:.4f}')
print(f'Precision = {prec_knn:.4f}')
print(f'Recall    = {rec_knn:.4f}')

cm_knn = confusion_matrix(y_test_c, y_pred_knn_c)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_knn,
                                display_labels=['Away Win', 'Draw', 'Home Win'])
fig, ax = plt.subplots(figsize=(7, 5))
disp2.plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('KNN Classifier — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== KNN REGRESSOR ===')
print(f'MAE  = {mae_knn:.4f}')
print(f'MSE  = {mse_knn:.4f}')
print(f'RMSE = {rmse_knn:.4f}')

### 4.4 — Model Comparison Summary

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'KNN Regressor', 'Logistic Regression', 'KNN Classifier'],
    'Type': ['Regression', 'Regression', 'Classification', 'Classification'],
    'MAE / Accuracy': [f'{mae_lr:.4f}', f'{mae_knn:.4f}', f'{acc_log:.4f}', f'{acc_knn:.4f}'],
    'MSE / Precision': [f'{mse_lr:.4f}', f'{mse_knn:.4f}', f'{prec_log:.4f}', f'{prec_knn:.4f}'],
    'RMSE / Recall': [f'{rmse_lr:.4f}', f'{rmse_knn:.4f}', f'{rec_log:.4f}', f'{rec_knn:.4f}']
})
print('=== MODEL COMPARISON ===')
comparison

---
## 🏆 Step 5: FIFA World Cup 2026 — Tournament Simulator

Using the **official 2026 World Cup groups** (draw held December 5, 2025) and our trained **Logistic Regression** model to predict match outcomes.

Group standings are ranked using a **custom sorting algorithm** based on:
1. Points → 2. Goal Difference → 3. Goals Scored → 4. Elo Rating (tiebreaker)

### 5.1 — Official Groups & Helper Functions

In [ ]:
# ── Official FIFA World Cup 2026 Groups (December 5, 2025 draw)
WC2026_GROUPS = {
    'Group A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'Group B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'Group C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'Group D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'Group E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'Group F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'Group G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'Group H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'Group I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'Group J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'Group K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'Group L': ['England', 'Croatia', 'Ghana', 'Panama']
}

# ── Predict a match using Logistic Regression
def predict_match(team_a, team_b, neutral=1):
    elo_a = latest_elos.get(team_a, 1500.0)
    elo_b = latest_elos.get(team_b, 1500.0)
    form_a = (elo_a - 1500) / 200.0
    form_b = (elo_b - 1500) / 200.0
    feats = pd.DataFrame([[elo_a, elo_b, elo_a - elo_b,
                           form_a, form_b, form_a - form_b, neutral]],
                         columns=FEATURES)
    feats_s = scaler.transform(feats)
    probs = log_reg.predict_proba(feats_s)[0]  # [P(Away Win), P(Draw), P(Home Win)]
    return probs, elo_a - elo_b

# ── Simulate a group match (returns goals and points)
def sim_group_match(team_a, team_b):
    probs, _ = predict_match(team_a, team_b)
    outcome = np.random.choice([0, 1, 2], p=probs)
    if outcome == 2:   # Team A wins
        g_a = int(np.random.poisson(1.8)) + 1
        g_b = max(0, int(np.random.poisson(0.8)))
        if g_a <= g_b: g_a = g_b + 1
        return g_a, g_b, 3, 0
    elif outcome == 0: # Team B wins
        g_a = max(0, int(np.random.poisson(0.8)))
        g_b = int(np.random.poisson(1.8)) + 1
        if g_b <= g_a: g_b = g_a + 1
        return g_a, g_b, 0, 3
    else:              # Draw
        g = max(0, int(np.random.poisson(1.1)))
        return g, g, 1, 1

# ── Simulate a knockout match (no draws — penalty shootout if needed)
def sim_knockout_match(team_a, team_b):
    probs, elo_diff = predict_match(team_a, team_b)
    outcome = np.random.choice([0, 1, 2], p=probs)
    if outcome == 2:
        g_a = int(np.random.poisson(1.8)) + 1
        g_b = max(0, int(np.random.poisson(0.8)))
        if g_a <= g_b: g_a = g_b + 1
        return team_a, g_a, g_b, None, None
    elif outcome == 0:
        g_a = max(0, int(np.random.poisson(0.8)))
        g_b = int(np.random.poisson(1.8)) + 1
        if g_b <= g_a: g_b = g_a + 1
        return team_b, g_a, g_b, None, None
    else:  # Draw → penalties
        g = max(0, int(np.random.poisson(1.1)))
        p_a = max(0.1, min(0.9, 0.5 + elo_diff / 800))
        winner = np.random.choice([team_a, team_b], p=[p_a, 1 - p_a])
        pen_w, pen_l = 5, 4
        return winner, g, g, (pen_w if winner == team_a else pen_l), (pen_l if winner == team_a else pen_w)

print('✅ World Cup 2026 groups and helper functions ready!')

### 5.2 — Custom Sorting Algorithm (Bubble Sort for Group Standings)

In [ ]:
def is_better(t1, t2):
    """Returns True if t1 ranks HIGHER than t2.
    Sorting criteria: 1.Points, 2.Goal Diff, 3.Goals Scored, 4.Elo
    """
    if t1['pts'] != t2['pts']:   return t1['pts'] > t2['pts']
    if t1['gd']  != t2['gd']:    return t1['gd']  > t2['gd']
    if t1['gf']  != t2['gf']:    return t1['gf']  > t2['gf']
    return t1['elo'] > t2['elo']

def bubble_sort_standings(standings):
    """Custom stable Bubble Sort — ranks teams from strongest to weakest."""
    teams = list(standings)
    n = len(teams)
    for i in range(n):
        for j in range(n - i - 1):
            if not is_better(teams[j], teams[j+1]):
                teams[j], teams[j+1] = teams[j+1], teams[j]
    return teams

print('✅ Custom bubble sort algorithm defined!')

### 5.3 — Simulate Group Stage

In [ ]:
np.random.seed(2026)  # reproducible results

group_results = {}

for grp_name, teams in WC2026_GROUPS.items():
    standings = {
        t: {'team': t, 'pts': 0, 'gf': 0, 'ga': 0, 'gd': 0,
            'elo': latest_elos.get(t, 1500.0)}
        for t in teams
    }
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            ta, tb = teams[i], teams[j]
            g_a, g_b, pts_a, pts_b = sim_group_match(ta, tb)
            for t, gf, ga, pts in [(ta, g_a, g_b, pts_a), (tb, g_b, g_a, pts_b)]:
                standings[t]['pts'] += pts
                standings[t]['gf']  += gf
                standings[t]['ga']  += ga
                standings[t]['gd']  += gf - ga

    sorted_standings = bubble_sort_standings(list(standings.values()))
    group_results[grp_name] = sorted_standings

# Display all group standings
print('=' * 55)
print('         FIFA WORLD CUP 2026 — GROUP STANDINGS')
print('=' * 55)
for grp_name, standings in group_results.items():
    print(f'\n{grp_name}')
    print(f'  {'Team':<25} {'Pts':>4} {'GF':>4} {'GA':>4} {'GD':>4}')
    print('  ' + '-'*45)
    for i, t in enumerate(standings):
        qual = '✅' if i < 2 else ('🔸' if i == 2 else '❌')
        print(f'  {qual} {t["team"]:<23} {t["pts"]:>4} {t["gf"]:>4} {t["ga"]:>4} {t["gd"]:>+4}')

### 5.4 — Select Qualified Teams (Sort 3rd-placed using Bubble Sort)

In [ ]:
winners    = [group_results[g][0]['team'] for g in WC2026_GROUPS]
runners_up = [group_results[g][1]['team'] for g in WC2026_GROUPS]
thirds     = [group_results[g][2] for g in WC2026_GROUPS]

# Use our custom bubble sort to rank 3rd-placed teams
sorted_thirds = bubble_sort_standings(thirds)
best_thirds = [t['team'] for t in sorted_thirds[:8]]

print('12 Group Winners:')
for i, t in enumerate(winners, 1): print(f'  {i:2}. {t}')
print('\n12 Runners-up:')
for i, t in enumerate(runners_up, 1): print(f'  {i:2}. {t}')
print('\n8 Best 3rd-placed teams (sorted by points, GD, GF, Elo):')
for i, t in enumerate(best_thirds, 1): print(f'  {i:2}. {t}')

### 5.5 — Simulate Knockout Stages

In [ ]:
def print_match(stage, ta, tb, winner, g_a, g_b, pa, pb):
    pen = f' (Pens: {pa}-{pb})' if pa is not None else ''
    marker_a = '🏆' if winner == ta else '   '
    marker_b = '🏆' if winner == tb else '   '
    print(f'  {marker_a} {ta:<22} {g_a} — {g_b} {tb}{pen}')

def simulate_stage(teams, stage_name):
    print(f'\n{'=' * 55}')
    print(f'  {stage_name}')
    print(f'{'=' * 55}')
    assert len(teams) % 2 == 0, 'Need even number of teams'
    winners = []
    for i in range(0, len(teams), 2):
        ta, tb = teams[i], teams[i+1]
        winner, g_a, g_b, pa, pb = sim_knockout_match(ta, tb)
        print_match(stage_name, ta, tb, winner, g_a, g_b, pa, pb)
        winners.append(winner)
    return winners

# Build Round of 32 bracket:
# 8 Winners vs 8 Best-thirds | 4 Winners vs 4 Runners-up | 4 Runners-up vs 4 Runners-up
r32 = []
for i in range(8):  r32 += [winners[i], best_thirds[i]]
for i in range(4):  r32 += [winners[8+i], runners_up[i]]
for i in range(4):  r32 += [runners_up[4+2*i], runners_up[5+2*i]]

r16_teams     = simulate_stage(r32, 'ROUND OF 32')
qf_teams      = simulate_stage(r16_teams, 'ROUND OF 16')
sf_teams      = simulate_stage(qf_teams, 'QUARTER-FINALS')
final_teams   = simulate_stage(sf_teams, 'SEMI-FINALS')

# 3rd place
losers = [t for t in sf_teams if t not in final_teams]
print(f'\n{'=' * 55}\n  THIRD PLACE PLAY-OFF\n{'=' * 55}')
w3, g_a, g_b, pa, pb = sim_knockout_match(losers[0], losers[1])
print_match('3rd', losers[0], losers[1], w3, g_a, g_b, pa, pb)

# Final
print(f'\n{'=' * 55}\n  *** THE FINAL ***\n{'=' * 55}')
champion, g_a, g_b, pa, pb = sim_knockout_match(final_teams[0], final_teams[1])
print_match('Final', final_teams[0], final_teams[1], champion, g_a, g_b, pa, pb)

### 5.6 — 🏆 Predicted World Cup 2026 Champion

In [ ]:
print('\n' + '🌟' * 25)
print(f'\n  🏆 FIFA WORLD CUP 2026 CHAMPION 🏆')
print(f'\n  ⚽  {champion.upper()}  ⚽')
print(f'\n  Predicted using: Logistic Regression ML Model')
print(f'  Based on {df_modern.shape[0]:,} historical matches (2000–2026)')
print('\n' + '🌟' * 25)